# 11 Evaluate Retrieval and QA

Σε αυτό το notebook αξιολογούνται τα retrieval και QA αποτελέσματα για τα `dense`, `hybrid` και `hybrid_reranked` pipelines. Η αξιολόγηση περιλαμβάνει retrieval metrics και ντετερμινιστικές μετρικές ποιότητας απάντησης.


In [ ]:
# Uncomment on Kaggle if needed:
# !pip install -q ragas datasets google-generativeai python-dotenv rapidfuzz

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

from scripts.evaluation_metrics import (
    context_support_score,
    evaluate_answer_record,
    exact_substring_match,
    lexical_f1,
    normalize_finance_answer,
    normalize_text,
    tokenize,
)


In [ ]:
RUN_NAMES = ["dense", "hybrid", "hybrid_reranked"]

USE_RAGAS = False
RAGAS_SAMPLE_LIMIT = None   # e.g. 30 for quick test, or None for full run

SAVE_OUTPUTS = True

# Always initialize so later cells can run even when RAGAS is disabled.
ragas_results_df = pd.DataFrame()

EVAL_CONFIG = {
    "run_names": RUN_NAMES,
    "use_ragas": USE_RAGAS,
    "ragas_sample_limit": RAGAS_SAMPLE_LIMIT,
}
EVAL_CONFIG

In [ ]:
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    BASE_DIR = CURRENT_DIR.parent
else:
    BASE_DIR = CURRENT_DIR

DATA_DIR = BASE_DIR / "data"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
QA_DIR = PROCESSED_DIR / "qa_results"
RETRIEVAL_DIR = PROCESSED_DIR / "retrieval_results"
EVAL_DIR = PROCESSED_DIR / "evaluation"

EVAL_DIR.mkdir(parents=True, exist_ok=True)

WORKING_DATASET_CSV_PATH = INTERIM_DIR / "financebench_open_source_working.csv"
WORKING_DATASET_PARQUET_PATH = INTERIM_DIR / "financebench_open_source_working.parquet"

DENSE_QA_CSV_PATH = QA_DIR / "rag_qa_results_dense.csv"
HYBRID_QA_CSV_PATH = QA_DIR / "rag_qa_results_hybrid.csv"
RERANK_QA_CSV_PATH = QA_DIR / "rag_qa_results_hybrid_reranked.csv"

DENSE_RETRIEVAL_CSV_PATH = RETRIEVAL_DIR / "retrieval_results_dense.csv"
HYBRID_RETRIEVAL_CSV_PATH = RETRIEVAL_DIR / "retrieval_results_hybrid.csv"
RERANK_RETRIEVAL_CSV_PATH = RETRIEVAL_DIR / "retrieval_results_hybrid_reranked.csv"

EVALUATION_SUMMARY_CSV_PATH = EVAL_DIR / "evaluation_summary.csv"
RETRIEVAL_COMPARISON_CSV_PATH = EVAL_DIR / "retrieval_evaluation_comparison.csv"
QA_COMPARISON_CSV_PATH = EVAL_DIR / "qa_evaluation_comparison.csv"
RETRIEVAL_PER_QUERY_CSV_PATH = EVAL_DIR / "retrieval_evaluation_per_query.csv"
QA_PER_QUERY_CSV_PATH = EVAL_DIR / "qa_evaluation_per_query.csv"
QA_PAIRWISE_CSV_PATH = EVAL_DIR / "qa_pairwise_comparison.csv"
QUALITATIVE_EXAMPLES_CSV_PATH = EVAL_DIR / "qualitative_examples.csv"
RAGAS_RESULTS_CSV_PATH = EVAL_DIR / "ragas_results.csv"
EVALUATION_STATS_JSON_PATH = EVAL_DIR / "evaluation_stats.json"

print("BASE_DIR:", BASE_DIR)
print("EVAL_DIR:", EVAL_DIR)

In [ ]:
if WORKING_DATASET_PARQUET_PATH.exists():
    try:
        working_df = pd.read_parquet(WORKING_DATASET_PARQUET_PATH)
    except ImportError:
        working_df = pd.read_csv(WORKING_DATASET_CSV_PATH)
elif WORKING_DATASET_CSV_PATH.exists():
    working_df = pd.read_csv(WORKING_DATASET_CSV_PATH)
else:
    raise FileNotFoundError("Working dataset not found.")

print("working_df shape:", working_df.shape)
print(working_df.columns.tolist())
working_df.head(2)

In [ ]:
qa_runs = {
    "dense": DENSE_QA_CSV_PATH,
    "hybrid": HYBRID_QA_CSV_PATH,
    "hybrid_reranked": RERANK_QA_CSV_PATH,
}

qa_dfs = {}

for run_name, path in qa_runs.items():
    if path.exists():
        qa_dfs[run_name] = pd.read_csv(path)
        print(f"{run_name}: {qa_dfs[run_name].shape}")
    else:
        print(f"{run_name}: το αρχείο δεν βρέθηκε -> {path}")


In [ ]:
retrieval_runs = {
    "dense": DENSE_RETRIEVAL_CSV_PATH,
    "hybrid": HYBRID_RETRIEVAL_CSV_PATH,
    "hybrid_reranked": RERANK_RETRIEVAL_CSV_PATH,
}

retrieval_dfs = {}

for run_name, path in retrieval_runs.items():
    if path.exists():
        retrieval_dfs[run_name] = pd.read_csv(path)
        print(f"{run_name}: {retrieval_dfs[run_name].shape}")
    else:
        print(f"{run_name}: το αρχείο δεν βρέθηκε -> {path}")


In [ ]:
# Finance-aware normalization and scoring are imported from
# scripts.evaluation_metrics so every downstream notebook uses one definition.


In [ ]:
# Text and numeric QA metrics use the shared publication evaluator.


In [ ]:
import ast, re

# Κατώφλι token-F1 πάνω από το οποίο ένα chunk θεωρείται ότι καλύπτει το evidence.
# Ακολουθεί την κλασική token-overlap F1 μετρική της extractive QA (SQuAD, Rajpurkar et al. 2016).
EVIDENCE_F1_THRESHOLD = 0.3

def _build_evidence_map(working_df):
    import numpy as np

    def get_ev(ev_val):
        # 1) Ήδη parsed list/array (συμβαίνει όταν φορτώνεται από parquet)
        if isinstance(ev_val, (list, tuple, np.ndarray)):
            items = list(ev_val)
        # 2) String που μοιάζει με list (συμβαίνει όταν φορτώνεται από CSV)
        elif isinstance(ev_val, str):
            try:
                items = ast.literal_eval(ev_val)
            except Exception:
                # fallback: χρησιμοποίησε το ίδιο το string ως evidence
                return ev_val
        else:
            return ""

        texts = []
        for it in items:
            if isinstance(it, dict):
                t = it.get("evidence_text", "")
                if t:
                    texts.append(str(t))
            elif it:
                texts.append(str(it))
        return " ".join(texts)

    return dict(zip(working_df["financebench_id"], working_df["evidence"].apply(get_ev)))

def _normalize(t):
    t = str(t).lower()
    t = re.sub(r"[^a-z0-9%\.\,\$\- ]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

def _token_f1(chunk_text, evidence_text):
    c = set(_normalize(chunk_text).split())
    e = set(_normalize(evidence_text).split())
    if not c or not e:
        return 0.0
    overlap = len(c & e)
    if overlap == 0:
        return 0.0
    p, r = overlap / len(c), overlap / len(e)
    return 2 * p * r / (p + r)

def evaluate_retrieval_run(run_name, retrieval_df, working_df):
    # evidence-level metrics: token-F1 overlap ανακτηθέντος chunk με το annotated evidence passage
    ev_map = _build_evidence_map(working_df)

    def _ev_match(row):
        ev = ev_map.get(row["financebench_id"], "")
        return _token_f1(row.get("chunk_text", ""), ev) >= EVIDENCE_F1_THRESHOLD
    retrieval_df["evidence_match"] = retrieval_df.apply(_ev_match, axis=1)

    unique_queries = retrieval_df["financebench_id"].nunique()

    def hit_at_k(df, k):
        topk = df[df["retrieved_rank"] <= k]
        return float(topk.groupby("financebench_id")["evidence_match"].max().mean())

    hit1 = hit_at_k(retrieval_df, 1)
    hit3 = hit_at_k(retrieval_df, 3)
    hit5 = hit_at_k(retrieval_df, 5)

    rr_values, best_scores, detail_rows = [], [], []
    for qid, group in retrieval_df.groupby("financebench_id"):
        group = group.sort_values("retrieved_rank")
        rr = 0.0
        for _, row in group.iterrows():
            if bool(row["evidence_match"]):
                rr = 1.0 / int(row["retrieved_rank"])
                break
        rr_values.append(rr)
        matching_ranks = group.loc[group["evidence_match"], "retrieved_rank"]
        first_evidence_rank = (
            int(matching_ranks.min()) if len(matching_ranks) else np.nan
        )
        top_row = group.iloc[0]
        detail_rows.append({
            "run_name": run_name,
            "financebench_id": qid,
            "question": top_row.get("question"),
            "expected_doc_name": top_row.get("expected_doc_name"),
            "top_doc_id": top_row.get("retrieved_doc_id"),
            "top_chunk_id": top_row.get("chunk_id"),
            "first_evidence_rank": first_evidence_rank,
            "evidence_hit_at_1": float(rr == 1.0),
            "evidence_hit_at_3": float(
                len(matching_ranks) and matching_ranks.min() <= 3
            ),
            "evidence_hit_at_5": float(
                len(matching_ranks) and matching_ranks.min() <= 5
            ),
            "reciprocal_rank": rr,
        })
        if "retrieval_score" in group.columns:
            best_scores.append(group["retrieval_score"].max())
        elif "rrf_score" in group.columns:
            best_scores.append(group["rrf_score"].max())
        else:
            best_scores.append(np.nan)

    summary = {
        "run_name": run_name,
        "n_queries": int(unique_queries),
        "evidence_hit_at_1": hit1,
        "evidence_hit_at_3": hit3,
        "evidence_hit_at_5": hit5,
        "mrr": float(np.mean(rr_values)) if rr_values else 0.0,
        "avg_best_match_score": float(np.nanmean(best_scores)) if best_scores else np.nan,
        "evaluation_type": "retrieval",
    }
    return summary, pd.DataFrame(detail_rows)

In [ ]:
def evaluate_qa_run(run_name, qa_df):
    df = qa_df.copy()
    if "expected_answer" not in df.columns:
        df["expected_answer"] = None
    if "context_text" not in df.columns:
        df["context_text"] = ""

    detail_rows = []
    for _, row in df.iterrows():
        metrics = evaluate_answer_record(
            prediction=row.get("generated_answer", ""),
            reference=row.get("expected_answer", ""),
            question=row.get("question", ""),
            context=row.get("context_text", ""),
        )
        detail_rows.append({
            "run_name": run_name,
            "financebench_id": row.get("financebench_id"),
            "question": row.get("question"),
            "expected_answer": row.get("expected_answer"),
            "generated_answer": row.get("generated_answer"),
            "expected_doc_name": row.get("expected_doc_name"),
            "top_doc_id": row.get("top_doc_id"),
            "top_chunk_id": row.get("top_chunk_id"),
            "doc_match": row.get("doc_match"),
            "generated_answer_len": len(str(row.get("generated_answer", "")).split()),
            **metrics,
        })

    detail_df = pd.DataFrame(detail_rows)
    numeric_rows = detail_df[detail_df["numeric_applicable"]].copy()
    summary = {
        "run_name": run_name,
        "n_queries": int(df["financebench_id"].nunique()),
        "n_numeric_queries": int(len(numeric_rows)),
        "avg_finance_aware_score": float(detail_df["finance_aware_score"].mean()),
        "numeric_answer_accuracy": (
            float(numeric_rows["numeric_match"].mean())
            if len(numeric_rows) else np.nan
        ),
        "avg_numeric_coverage": (
            float(numeric_rows["numeric_coverage"].mean())
            if len(numeric_rows) else np.nan
        ),
        "avg_lexical_f1": float(detail_df["lexical_f1"].mean()),
        "exact_substring_match_rate": float(detail_df["exact_substring_match"].mean()),
        "avg_context_support_score": float(detail_df["context_support_score"].mean()),
        "insufficient_evidence_rate": float(detail_df["insufficient_evidence"].mean()),
        "avg_generated_answer_len": float(detail_df["generated_answer_len"].mean()),
        "evaluation_type": "qa_finance_aware",
    }
    return summary, detail_df

In [ ]:
retrieval_eval_rows = []
retrieval_detail_frames = []

for run_name, df in retrieval_dfs.items():
    summary, detail = evaluate_retrieval_run(run_name, df.copy(), working_df)
    retrieval_eval_rows.append(summary)
    retrieval_detail_frames.append(detail)

retrieval_eval_df = pd.DataFrame(retrieval_eval_rows)
retrieval_per_query_df = pd.concat(retrieval_detail_frames, ignore_index=True)
retrieval_eval_df

In [ ]:
qa_eval_rows = []
qa_detail_frames = []

for run_name, df in qa_dfs.items():
    summary, detail = evaluate_qa_run(run_name, df)
    qa_eval_rows.append(summary)
    qa_detail_frames.append(detail)

qa_eval_df = pd.DataFrame(qa_eval_rows)
qa_per_query_df = pd.concat(qa_detail_frames, ignore_index=True)

pairwise_metric_columns = [
    "finance_aware_score", "numeric_match", "numeric_coverage",
    "lexical_f1", "context_support_score", "insufficient_evidence",
]
qa_pairwise_df = qa_per_query_df[
    ["financebench_id", "run_name", *pairwise_metric_columns]
].pivot(index="financebench_id", columns="run_name")
qa_pairwise_df.columns = [
    f"{run_name}_{metric}" for metric, run_name in qa_pairwise_df.columns
]
qa_pairwise_df = qa_pairwise_df.reset_index()

qa_eval_df

In [ ]:
qualitative_rows = []

all_ids = sorted(set(working_df["financebench_id"].tolist()))

for financebench_id in all_ids:
    base_row = working_df[working_df["financebench_id"] == financebench_id].head(1)
    if len(base_row) == 0:
        continue

    row = base_row.iloc[0]
    out = {
        "financebench_id": financebench_id,
        "question": row.get("question"),
        "gold_answer": row.get("answer"),
        "expected_doc_name": row.get("doc_name"),
    }

    for run_name, df in qa_dfs.items():
        match = df[df["financebench_id"] == financebench_id].head(1)
        if len(match):
            m = match.iloc[0]
            out[f"{run_name}_answer"] = m.get("generated_answer")
            out[f"{run_name}_top_doc"] = m.get("top_doc_id")
            out[f"{run_name}_top_chunk"] = m.get("top_chunk_id")
            out[f"{run_name}_doc_match"] = m.get("doc_match")
        else:
            out[f"{run_name}_answer"] = None
            out[f"{run_name}_top_doc"] = None
            out[f"{run_name}_top_chunk"] = None
            out[f"{run_name}_doc_match"] = None

    qualitative_rows.append(out)

qualitative_examples_df = pd.DataFrame(qualitative_rows)
qualitative_examples_df.head(10)

In [ ]:
# Build one export-friendly summary table from all available evaluation layers.
summary_frames = [
    retrieval_eval_df.assign(metric_group="retrieval"),
    qa_eval_df.assign(metric_group="qa_deterministic"),
]

if len(ragas_results_df):
    summary_frames.append(ragas_results_df.assign(metric_group="ragas"))

evaluation_summary_df = pd.concat(summary_frames, ignore_index=True, sort=False)

if SAVE_OUTPUTS:
    retrieval_eval_df.to_csv(RETRIEVAL_COMPARISON_CSV_PATH, index=False, encoding="utf-8")
    retrieval_per_query_df.to_csv(RETRIEVAL_PER_QUERY_CSV_PATH, index=False, encoding="utf-8")
    qa_eval_df.to_csv(QA_COMPARISON_CSV_PATH, index=False, encoding="utf-8")
    qa_per_query_df.to_csv(QA_PER_QUERY_CSV_PATH, index=False, encoding="utf-8")
    qa_pairwise_df.to_csv(QA_PAIRWISE_CSV_PATH, index=False, encoding="utf-8")
    qualitative_examples_df.to_csv(QUALITATIVE_EXAMPLES_CSV_PATH, index=False, encoding="utf-8")
    evaluation_summary_df.to_csv(EVALUATION_SUMMARY_CSV_PATH, index=False, encoding="utf-8")

    if len(ragas_results_df):
        ragas_results_df.to_csv(RAGAS_RESULTS_CSV_PATH, index=False, encoding="utf-8")

    print("Saved evaluation outputs.")
    print("-", RETRIEVAL_COMPARISON_CSV_PATH)
    print("-", RETRIEVAL_PER_QUERY_CSV_PATH)
    print("-", QA_COMPARISON_CSV_PATH)
    print("-", QA_PER_QUERY_CSV_PATH)
    print("-", QA_PAIRWISE_CSV_PATH)
    print("-", QUALITATIVE_EXAMPLES_CSV_PATH)
    print("-", EVALUATION_SUMMARY_CSV_PATH)
    if len(ragas_results_df):
        print("-", RAGAS_RESULTS_CSV_PATH)

evaluation_summary_df

In [ ]:
evaluation_stats = {
    "n_runs": len(RUN_NAMES),
    "runs": RUN_NAMES,
    "use_ragas": USE_RAGAS,
    "ragas_sample_limit": RAGAS_SAMPLE_LIMIT,
    "retrieval_comparison_csv": str(RETRIEVAL_COMPARISON_CSV_PATH),
    "qa_comparison_csv": str(QA_COMPARISON_CSV_PATH),
    "retrieval_per_query_csv": str(RETRIEVAL_PER_QUERY_CSV_PATH),
    "qa_per_query_csv": str(QA_PER_QUERY_CSV_PATH),
    "qa_pairwise_csv": str(QA_PAIRWISE_CSV_PATH),
    "qualitative_examples_csv": str(QUALITATIVE_EXAMPLES_CSV_PATH),
    "evaluation_summary_csv": str(EVALUATION_SUMMARY_CSV_PATH),
}

with open(EVALUATION_STATS_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(evaluation_stats, f, indent=2, ensure_ascii=False)

print("Saved stats:", EVALUATION_STATS_JSON_PATH)
evaluation_stats

In [ ]:
print("=== Retrieval Comparison ===")
display(retrieval_eval_df)

print("=== QA Deterministic Comparison ===")
display(qa_eval_df)

if len(ragas_results_df):
    print("=== RAGAS Comparison ===")
    display(ragas_results_df)

print("=== Qualitative Examples ===")
display(qualitative_examples_df.head(10))